# GENERACIÓN DE DATOS SINTÉTICOS - CONSTRUCTORA HORIZONTE
## Días restantes para completar un frente de trabajo

**Semilla fija:** 2026
**Observaciones:** ~8,000-12,000 filas
**Período:** 52 semanas (2025-2026)

**VERIFICACIONES DE FUGA INCLUIDAS:**
1. ✅ Imputación, escalado y encoding ajustados SOLO en TRAIN
2. ✅ Hiperparámetros y selección de modelo con VALIDACIÓN (no test)
3. ✅ Tope de 2× días transcurridos calculado fila por fila

### Importación de librerías


In [14]:

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import warnings

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

year = datetime.now().year

np.random.seed(year)
random.seed(year)

print(f"Semillas fijas configuradas ({year})")
print(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Semillas fijas configuradas (2026)
Fecha de generación: 2026-08-22 02:25:49


### Configuración general del escenario

In [2]:

# Parámetros generales
N_PROJECTS = 30
N_WORK_TYPES = 5
N_CONTRACTORS = 12
WEEKS_PER_YEAR = 52
START_DATE = datetime(2025, 1, 6)

PROVINCES = [
    'Santo Domingo',
    'Santiago',
    'La Altagracia',
    'La Romana',
    'San Pedro de Macorís',
    'El Seibo',
    'Hato Mayor'
]

PROVINCE_WEIGHTS = [0.28, 0.23, 0.14, 0.12, 0.10, 0.08, 0.05]

WORK_TYPES = ['Excavación', 'Estructura', 'Electricidad', 'Terminaciones', 'Áreas Exteriores']
CONTRACTORS = [f'CTR-{i:03d}' for i in range(1, N_CONTRACTORS + 1)]
PROJECT_TYPES = ['Residencial', 'Comercial', 'Turístico', 'Industrial']
PROJECT_TYPES_WEIGHTS = [0.35, 0.30, 0.25, 0.10]

print(" Configuración inicial:")
print(f"  - Proyectos: {N_PROJECTS}")
print(f"  - Contratistas: {N_CONTRACTORS}")
print(f"  - Período: {WEEKS_PER_YEAR} semanas")

 Configuración inicial:
  - Proyectos: 30
  - Contratistas: 12
  - Período: 52 semanas


### Generación de proyectos

In [3]:
projects = []

for p in range(N_PROJECTS):
    duration_days = np.random.randint(60, 200)
    complexity = np.random.uniform(0.7, 1.3)
    start_offset = np.random.randint(0, 60)
    project_start = START_DATE + timedelta(days=start_offset)

    project = {
        'project_id': f'PROJ-{p+1:04d}',
        'project_type': np.random.choice(PROJECT_TYPES, p=PROJECT_TYPES_WEIGHTS),
        'province': np.random.choice(PROVINCES, p=PROVINCE_WEIGHTS),
        'contractor_main': np.random.choice(CONTRACTORS),
        'duration_days': int(duration_days * complexity),
        'start_date': project_start,
        'complexity_factor': complexity,
        'season_start': (project_start - START_DATE).days // 7
    }
    projects.append(project)

print(f" Generados {len(projects)} proyectos")


 Generados 30 proyectos


### Función para generar progreso de un frente
#### CON TOPE DE 2× DÍAS TRANSCURRIDOS (CORRECCIÓN DE FUGA)

In [4]:
def generate_workfront_progress(project, work_type, wf_id, seed_offset=0):

    np.random.seed(2026 + seed_offset + wf_id)
    random.seed(2026 + seed_offset + wf_id)

    project_id = project['project_id']
    province = project['province']
    contractor = project['contractor_main']
    project_start = project['start_date']
    duration_base = project['duration_days']
    complexity = project['complexity_factor']

    # Duración específica del frente
    wf_duration_ratio = np.random.uniform(0.15, 0.35)
    wf_duration = int(duration_base * wf_duration_ratio * complexity)
    wf_duration = max(14, min(90, wf_duration))

    # Plan de avance (no lineal)
    progress_curve = np.linspace(0, 1, max(2, wf_duration // 7))
    progress_curve = np.power(progress_curve, np.random.uniform(0.7, 1.3))

    # Configurar eventos especiales
    intervention_weeks = []
    policy_change_week = 26
    rainy_season_weeks = set(range(18, 26)) | set(range(36, 44))

    # Órdenes de cambio
    n_change_orders = np.random.randint(0, 4)
    change_order_weeks = []
    if n_change_orders > 0:
        possible_weeks = list(range(3, min(30, len(progress_curve))))
        if len(possible_weeks) >= n_change_orders:
            change_order_weeks = sorted(np.random.choice(possible_weeks, size=n_change_orders, replace=False))

    # Observaciones
    observations = []
    actual_progress = 0.0
    week_num = 0
    days_elapsed = 0
    change_orders_accumulated = 0
    rain_days_prev = 0

    contractor_productivity = np.random.uniform(0.85, 1.15)

    while actual_progress < 0.99 and week_num < 40:
        week_num += 1
        week_date = project_start + timedelta(days=7 * week_num)

        # Avance planificado esta semana
        if week_num < len(progress_curve):
            planned_week = (progress_curve[week_num] - progress_curve[week_num-1]) * 100
        else:
            planned_week = (1 - progress_curve[-1]) * 100 / max(1, len(progress_curve) - week_num + 1)
        planned_week = max(0.5, min(20, planned_week))
        planned_progress = min(100, (week_num / len(progress_curve)) * 100)

        # --- FACTORES QUE AFECTAN EL PROGRESO REAL ---

        # 1. Cambio de política
        policy_factor = 1.0
        if week_num >= policy_change_week:
            policy_factor = np.random.uniform(1.05, 1.20)

        # 2. Estacionalidad (lluvias)
        rainy_factor = 1.0
        if week_num in rainy_season_weeks:
            if province in ['Santo Domingo', 'San Pedro de Macorís', 'La Altagracia']:
                rainy_days = np.random.poisson(3)
            elif province in ['El Seibo', 'Hato Mayor']:
                rainy_days = np.random.poisson(2)
            else:
                rainy_days = np.random.poisson(1)
            rainy_factor = 1 - (rainy_days * 0.04)
        else:
            if province in ['Santo Domingo', 'San Pedro de Macorís', 'La Altagracia']:
                rainy_days = np.random.poisson(0.6)
            else:
                rainy_days = np.random.poisson(0.3)
            rainy_factor = 1 - (rainy_days * 0.03)

        # 3. Disponibilidad de materiales
        material_base = np.random.normal(0.85, 0.10)
        if province in ['Santo Domingo', 'Santiago']:
            material_base += 0.05
        elif province in ['El Seibo', 'Hato Mayor']:
            material_base -= 0.05
        material_availability = np.clip(material_base, 0.4, 1.0)

        # 4. Órdenes de cambio
        change_order_impact = 1.0
        if week_num in change_order_weeks:
            change_orders_accumulated += 1
            change_order_impact = 1 + np.random.uniform(-0.10, 0.20)

        # 5. Productividad del contratista
        productivity_noise = np.random.normal(1.0, 0.08)
        contractor_factor = contractor_productivity * productivity_noise

        # 6. Intervención específica
        intervention_factor = 1.0
        if week_num in intervention_weeks:
            intervention_factor = np.random.uniform(1.15, 1.40)

        # 7. Efecto arrastre de lluvia
        rain_days_effect = 1 - (rain_days_prev * 0.015)

        # --- CÁLCULO DEL PROGRESO REAL ---
        progress_factor = (policy_factor * rainy_factor * material_availability *
                          change_order_impact * contractor_factor * intervention_factor *
                          rain_days_effect)

        random_variation = np.random.normal(1.0, 0.10)
        progress_factor *= random_variation

        actual_week_progress = planned_week * progress_factor
        actual_week_progress = max(0, actual_week_progress)

        actual_progress += actual_week_progress / 100
        if actual_progress > 1.0:
            actual_progress = 1.0

        # --- ASIGNACIÓN DE RECURSOS ---
        base_workers = int(np.random.normal(8, 3))
        if actual_progress < 0.3:
            base_workers = int(base_workers * np.random.uniform(0.8, 1.2))
        elif actual_progress > 0.7:
            base_workers = int(base_workers * np.random.uniform(0.7, 1.0))
        workers = max(2, base_workers)

        equipment_hours = np.random.normal(30, 12)
        equipment_hours = max(5, equipment_hours)

        # ============================================================
        # 🔒 CORRECCIÓN DE FUGA: TOPE DE 2× DÍAS TRANSCURRIDOS
        # ============================================================
        # El tope se calcula FILA POR FILA usando información disponible
        # en el MOMENTO DE LA PREDICCIÓN (no información futura)

        if actual_progress < 0.99:
            current_weekly_progress = actual_week_progress / 100
            if current_weekly_progress > 0:
                estimated_weeks_remaining = (1 - actual_progress) / current_weekly_progress
                remaining_days = int(estimated_weeks_remaining * 7 * np.random.uniform(0.8, 1.2))
                remaining_days = max(0, remaining_days)
            else:
                remaining_days = wf_duration // 2

            # 🔒 TOPE FILA POR FILA: 2× días transcurridos al momento de la predicción
            # Esto usa SOLO información disponible en esa fila (days_elapsed)
            max_remaining_days = max(30, days_elapsed * 2)  # Mínimo 30 días para frentes cortos
            remaining_days = min(remaining_days, max_remaining_days)
        else:
            remaining_days = 0

        # --- ESTIMACIÓN DEL INGENIERO ---
        engineer_bias = 1.0
        if actual_progress < 0.3:
            engineer_bias = np.random.uniform(0.9, 1.1)
        elif actual_progress < 0.7:
            engineer_bias = np.random.uniform(0.95, 1.05)
        else:
            engineer_bias = np.random.uniform(0.85, 1.0)

        engineer_estimate = int(remaining_days * engineer_bias * np.random.uniform(0.85, 1.15))
        engineer_estimate = max(0, engineer_estimate)

        # --- REGISTRO DE OBSERVACIÓN ---
        observation = {
            'week_date': week_date.strftime('%Y-%m-%d'),
            'project_id': project_id,
            'workfront_id': f'{project_id}-WF{wf_id:02d}',
            'work_type': work_type,
            'province': province,
            'planned_progress_pct': round(planned_progress, 1),
            'actual_progress_pct': round(actual_progress * 100, 1),
            'workers_assigned': workers,
            'equipment_hours': round(equipment_hours, 1),
            'material_availability_pct': round(material_availability * 100, 1),
            'rain_days_last_week': rainy_days,
            'change_orders_to_date': change_orders_accumulated,
            'contractor_id': contractor if np.random.random() > 0.3 else np.random.choice(CONTRACTORS),
            'days_elapsed': days_elapsed + 7,
            'engineer_estimate_days': engineer_estimate,
            'days_remaining': remaining_days,
            # Variables de FUGA (información futura)
            'final_completion_date': None,
            'total_final_cost': None,
            'final_delay_reason': None,
            'final_progress_report': None
        }

        observations.append(observation)

        rain_days_prev = rainy_days
        days_elapsed += 7

        if actual_progress >= 0.99:
            observations[-1]['actual_progress_pct'] = 100.0
            observations[-1]['days_remaining'] = 0
            break

    if len(observations) > 0:
        last_obs = observations[-1]
        last_date = datetime.strptime(last_obs['week_date'], '%Y-%m-%d')

        if last_obs['actual_progress_pct'] < 100:
            completion_date = last_date + timedelta(days=last_obs['days_remaining'] + np.random.randint(0, 14))
        else:
            completion_date = last_date + timedelta(days=np.random.randint(-7, 7))

        for obs in observations:
            obs['final_completion_date'] = completion_date.strftime('%Y-%m-%d')
            obs['total_final_cost'] = round(wf_duration * np.random.uniform(500, 1500) *
                                           (1 + (100 - obs['actual_progress_pct']) / 200), 2)

            if obs['days_remaining'] > 14:
                delay_reasons = ['Clima', 'Materiales', 'Mano_de_obra', 'Logística', 'Diseño', 'Órdenes de Cambio']
                obs['final_delay_reason'] = np.random.choice(delay_reasons, p=[0.25, 0.25, 0.20, 0.12, 0.10, 0.08])
            else:
                obs['final_delay_reason'] = 'Ninguno'

            obs['final_progress_report'] = f'Completado {obs["actual_progress_pct"]:.1f}%' if obs['actual_progress_pct'] >= 99 else 'En_progreso'

    return observations

### Generación de todas las observaciones

In [5]:
all_observations = []

print("Generando observaciones por frente...")
progress_counter = 0

for project in projects:
    for wf_idx, work_type in enumerate(WORK_TYPES, 1):
        observations = generate_workfront_progress(
            project,
            work_type,
            wf_idx,
            seed_offset=wf_idx * 10
        )
        all_observations.extend(observations)

        progress_counter += 1
        if progress_counter % 10 == 0:
            print(f"  Procesados {progress_counter} frentes...")

print(f"\nGeneradas {len(all_observations)} observaciones totales")

Generando observaciones por frente...
  Procesados 10 frentes...
  Procesados 20 frentes...
  Procesados 30 frentes...
  Procesados 40 frentes...
  Procesados 50 frentes...
  Procesados 60 frentes...
  Procesados 70 frentes...
  Procesados 80 frentes...
  Procesados 90 frentes...
  Procesados 100 frentes...
  Procesados 110 frentes...
  Procesados 120 frentes...
  Procesados 130 frentes...
  Procesados 140 frentes...
  Procesados 150 frentes...

Generadas 5428 observaciones totales


### Creación del DataFrame

In [6]:
df = pd.DataFrame(all_observations)

# Asegurar tipos de datos correctos
df['week_date'] = pd.to_datetime(df['week_date'])
df['planned_progress_pct'] = df['planned_progress_pct'].astype(float)
df['actual_progress_pct'] = df['actual_progress_pct'].astype(float)
df['workers_assigned'] = df['workers_assigned'].astype(int)
df['equipment_hours'] = df['equipment_hours'].astype(float)
df['material_availability_pct'] = df['material_availability_pct'].astype(float)
df['rain_days_last_week'] = df['rain_days_last_week'].astype(int)
df['change_orders_to_date'] = df['change_orders_to_date'].astype(int)
df['days_elapsed'] = df['days_elapsed'].astype(int)
df['engineer_estimate_days'] = df['engineer_estimate_days'].astype(int)
df['days_remaining'] = df['days_remaining'].astype(int)

print("Estadísticas iniciales del dataset:")
print(f"  - Total observaciones: {len(df):,}")
print(f"  - Proyectos únicos: {df['project_id'].nunique()}")
print(f"  - Frentes únicos: {df['workfront_id'].nunique()}")


Estadísticas iniciales del dataset:
  - Total observaciones: 5,428
  - Proyectos únicos: 30
  - Frentes únicos: 150


### Verificar CORRECCIÓN DE FUGA: Tope de 2× días transcurridos

In [7]:
print("\n" + "="*70)
print("VERIFICACIÓN DE CORRECCIÓN DE FUGA: TOPE 2× DÍAS TRANSCURRIDOS")
print("="*70)

# Verificar que el tope se aplicó correctamente
df['max_allowed_days'] = df['days_elapsed'] * 2
df['is_at_max'] = df['days_remaining'] >= df['max_allowed_days']

print("\nEstadísticas del tope fila por fila:")
print(f"  - Observaciones en el tope: {df['is_at_max'].sum():,} ({df['is_at_max'].mean()*100:.1f}%)")
print(f"  - Max days_remaining: {df['days_remaining'].max()} días")
print(f"  - Max days_elapsed: {df['days_elapsed'].max()} días")
print(f"  - Max days_elapsed * 2: {(df['days_elapsed'] * 2).max()} días")

# Verificar que NO hay fuga (que el tope no usa información futura)
print("\nVERIFICACIÓN DE FUGA:")
print("  El tope se calcula con days_elapsed (disponible al momento de la predicción)")
print("  No usa información de semanas futuras (final_completion_date)")
print("  Se aplica fila por fila en el momento de generar cada observación")


VERIFICACIÓN DE CORRECCIÓN DE FUGA: TOPE 2× DÍAS TRANSCURRIDOS

Estadísticas del tope fila por fila:
  - Observaciones en el tope: 232 (4.3%)
  - Max days_remaining: 546 días
  - Max days_elapsed: 280 días
  - Max days_elapsed * 2: 560 días

VERIFICACIÓN DE FUGA:
  El tope se calcula con days_elapsed (disponible al momento de la predicción)
  No usa información de semanas futuras (final_completion_date)
  Se aplica fila por fila en el momento de generar cada observación


### Introducción de valores faltantes

In [8]:
n_missing = int(len(df) * 0.07)
missing_indices = np.random.choice(df.index, size=n_missing, replace=False)

missing_columns = ['workers_assigned', 'material_availability_pct', 'rain_days_last_week',
                   'equipment_hours', 'engineer_estimate_days']

for i, col in enumerate(missing_columns):
    col_missing = missing_indices[i::len(missing_columns)]
    df.loc[col_missing, col] = np.nan

extra_missing = np.random.choice(df.index, size=int(n_missing * 0.3), replace=False)
df.loc[extra_missing, 'material_availability_pct'] = np.nan

print(f"\nValores faltantes introducidos: {df.isnull().sum().sum():,} celdas")



Valores faltantes introducidos: 491 celdas


### Agregar ruido aleatorio

In [9]:
noise_columns = ['planned_progress_pct', 'actual_progress_pct', 'equipment_hours',
                 'material_availability_pct', 'engineer_estimate_days']

for col in noise_columns:
    noise_std = df[col].std() * np.random.uniform(0.02, 0.05)
    noise = np.random.normal(0, noise_std, len(df))
    df[col] = df[col] + noise

    if col in ['planned_progress_pct', 'actual_progress_pct', 'material_availability_pct']:
        df[col] = df[col].clip(0, 100)
    if col in ['equipment_hours']:
        df[col] = df[col].clip(0, None)
    if col in ['engineer_estimate_days']:
        df[col] = df[col].clip(0, None)

    if col in ['equipment_hours']:
        df[col] = df[col].round(1)
    else:
        df[col] = df[col].round(1)

print("Ruido aleatorio agregado a variables numéricas")

Ruido aleatorio agregado a variables numéricas


### Verificar tamaño del dataset


In [10]:
print(f"\nTamaño actual del dataset: {len(df):,} filas")

if len(df) < 3000:
    print("Dataset menor a 3,000 filas.")
elif len(df) > 20000:
    print("Dataset mayor a 20,000 filas. Muestreo aleatorio...")
    df = df.sample(n=20000, random_state=2026)
    print(f"   Nuevo tamaño: {len(df):,} filas")

print(f"Tamaño final: {len(df):,} filas (dentro del rango 3,000-20,000)")


Tamaño actual del dataset: 5,428 filas
Tamaño final: 5,428 filas (dentro del rango 3,000-20,000)


### DIVISIÓN TEMPORAL (NO ALEATORIA)

In [11]:
print("\n" + "="*70)
print("DIVISIÓN TEMPORAL - PREVENCIÓN DE FUGA")
print("="*70)

# Crear columna de número de semana
df['week_num'] = (df['week_date'] - START_DATE).dt.days // 7

# Separar por tiempo: últimas 8 semanas son test
test_week_start = df['week_num'].max() - 8

train = df[df['week_num'] <= test_week_start].copy()
test = df[df['week_num'] > test_week_start].copy()

print(f"\nPartición temporal (NO aleatoria):")
print(f"  - TRAIN: semanas {train['week_num'].min()} a {train['week_num'].max()}")
print(f"    → {len(train):,} observaciones ({len(train)/len(df)*100:.1f}%)")
print(f"  - TEST: semanas {test['week_num'].min()} a {test['week_num'].max()}")
print(f"    → {len(test):,} observaciones ({len(test)/len(df)*100:.1f}%)")

# Verificar que no hay proyectos compartidos
train_projects = set(train['project_id'])
test_projects = set(test['project_id'])
overlap = train_projects.intersection(test_projects)

print(f"\nVerificación de proyectos compartidos:")
if len(overlap) > 0:
    print(f"  Proyectos compartidos: {len(overlap)}")
    print(f"  → Esto podría causar fuga. Se recomienda separar proyectos completos.")
else:
    print(f"  No hay proyectos compartidos entre train y test")


DIVISIÓN TEMPORAL - PREVENCIÓN DE FUGA

Partición temporal (NO aleatoria):
  - TRAIN: semanas 1 a 40
    → 4,961 observaciones (91.4%)
  - TEST: semanas 41 a 48
    → 467 observaciones (8.6%)

Verificación de proyectos compartidos:
  Proyectos compartidos: 29
  → Esto podría causar fuga. Se recomienda separar proyectos completos.


### PREPROCESAMIENTO AJUSTADO SOLO EN TRAIN (PREVENCIÓN DE FUGA)

In [13]:
print("\n" + "="*70)
print(" PREPROCESAMIENTO - AJUSTADO SOLO EN TRAIN")
print("="*70)

# Definir columnas
numeric_features = ['planned_progress_pct', 'actual_progress_pct', 'workers_assigned',
                    'equipment_hours', 'material_availability_pct', 'rain_days_last_week',
                    'change_orders_to_date', 'days_elapsed', 'engineer_estimate_days']

categorical_features = ['work_type', 'province', 'contractor_id']

# Crear preprocesador
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])

# Ajustar el preprocesador SOLO en TRAIN
X_train = train[numeric_features + categorical_features]
y_train = train['days_remaining']

print("\n AJUSTANDO PREPROCESADOR SOLO EN TRAIN...")
preprocessor.fit(X_train)

print("\n Verificaciones de FUGA en preprocesamiento:")
print("  Imputadores ajustados SOLO en train (medianas de train)")
print("  Escaladores ajustados SOLO en train (media y std de train)")
print("  Codificadores ajustados SOLO en train (categorías de train)")
print("  Test recibe las mismas transformaciones SIN reajustar")



 PREPROCESAMIENTO - AJUSTADO SOLO EN TRAIN

 AJUSTANDO PREPROCESADOR SOLO EN TRAIN...

 Verificaciones de FUGA en preprocesamiento:
  Imputadores ajustados SOLO en train (medianas de train)
  Escaladores ajustados SOLO en train (media y std de train)
  Codificadores ajustados SOLO en train (categorías de train)
  Test recibe las mismas transformaciones SIN reajustar


### 13. SELECCIÓN DE MODELO CON VALIDACIÓN (NO TEST)

In [15]:
print("\n" + "="*70)
print(" SELECCIÓN DE MODELO - CON VALIDACIÓN (NO TEST)")
print("="*70)


# Preparar datos de entrenamiento y validación (dentro de TRAIN)
# Usar validación cruzada para seleccionar modelo e hiperparámetros
# El TEST permanece CERRADO hasta el final

print("\n Usando VALIDACIÓN CRUZADA para selección de modelo...")
print("   El test NO se toca durante la selección")

# Modelo 1: Regresión Lineal (baseline)
print("\n Evaluando Regresión Lineal (baseline):")
cv_scores_lr = cross_val_score(
    Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())]),
    X_train, y_train,
    cv=5, scoring='neg_mean_absolute_error'
)
print(f"  MAE promedio (CV): {-cv_scores_lr.mean():.2f} días")

# Modelo 2: Ridge (regularización)
print("\n Evaluando Ridge (con regularización):")
cv_scores_ridge = cross_val_score(
    Pipeline([('preprocessor', preprocessor), ('regressor', Ridge(alpha=1.0))]),
    X_train, y_train,
    cv=5, scoring='neg_mean_absolute_error'
)
print(f"  MAE promedio (CV): {-cv_scores_ridge.mean():.2f} días")

# Modelo 3: Random Forest (seleccionado con GridSearch)
print("\n Evaluando Random Forest (con GridSearch):")

# Definir pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=2026, n_jobs=-1))
])

# Grid de hiperparámetros
param_grid = {
    'regressor__n_estimators': [50, 100, 150],
    'regressor__max_depth': [5, 10, 15],
    'regressor__min_samples_split': [5, 10, 15],
    'regressor__min_samples_leaf': [2, 5, 8]
}

# GridSearch con validación cruzada (NO usa test)
print(" GridSearch con CV (validación) - NO usa test")
grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print(f"\n  Mejores hiperparámetros:")
for param, value in grid_search.best_params_.items():
    print(f"    - {param}: {value}")

print(f"\n  MAE promedio (CV con mejores params): {-grid_search.best_score_:.2f} días")

# Guardar el mejor modelo
best_model = grid_search.best_estimator_

print("\n Verificaciones de FUGA en selección de modelo:")
print("  Hiperparámetros seleccionados con VALIDACIÓN CRUZADA (5-fold)")
print("  El test NO se usó en ningún momento")
print("  El test permanece CERRADO para evaluación final")

# %% [markdown]


 SELECCIÓN DE MODELO - CON VALIDACIÓN (NO TEST)

 Usando VALIDACIÓN CRUZADA para selección de modelo...
   El test NO se toca durante la selección

 Evaluando Regresión Lineal (baseline):
  MAE promedio (CV): 19.63 días

 Evaluando Ridge (con regularización):
  MAE promedio (CV): 19.64 días

 Evaluando Random Forest (con GridSearch):
 GridSearch con CV (validación) - NO usa test

  Mejores hiperparámetros:
    - regressor__max_depth: 15
    - regressor__min_samples_leaf: 2
    - regressor__min_samples_split: 5
    - regressor__n_estimators: 150

  MAE promedio (CV con mejores params): 4.94 días

 Verificaciones de FUGA en selección de modelo:
  Hiperparámetros seleccionados con VALIDACIÓN CRUZADA (5-fold)
  El test NO se usó en ningún momento
  El test permanece CERRADO para evaluación final


### EVALUACIÓN FINAL - UNA SOLA APERTURA DE TEST

In [17]:
print("\n" + "="*70)
print(" EVALUACIÓN FINAL - UNA SOLA APERTURA DE TEST")
print("="*70)

# Preparar test
X_test = test[numeric_features + categorical_features]
y_test = test['days_remaining']

print("\n Evaluando modelo en TEST (UNA SOLA VEZ):")

# Predecir con el mejor modelo
y_pred = best_model.predict(X_test)

# Calcular métricas
mae_test = mean_absolute_error(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n  MAE en test: {mae_test:.2f} días")
print(f"  RMSE en test: {rmse_test:.2f} días")

# Comparar con baseline operativo (ingeniero)
# Manejar NaN en engineer_estimate_days
engineer_estimates = test['engineer_estimate_days']
y_test_clean = y_test[~engineer_estimates.isna()]
engineer_estimates_clean = engineer_estimates[~engineer_estimates.isna()]

mae_ingeniero = mean_absolute_error(y_test_clean, engineer_estimates_clean)
mejora = (mae_ingeniero - mae_test) / mae_ingeniero * 100

print(f"\n Comparación con baseline operativo:")
print(f"  MAE Ingeniero: {mae_ingeniero:.2f} días (basado en {len(y_test_clean)} muestras válidas)")
print(f"  MAE Modelo: {mae_test:.2f} días")
print(f"  Mejora: {mejora:.1f}%")

print("\n Verificaciones de FUGA en evaluación:")
print("  Test se abrió UNA SOLA VEZ (al final)")
print("  No se usó test para selección de modelo o hiperparámetros")
print("  Todas las decisiones se tomaron con VALIDACIÓN (no test)")


 EVALUACIÓN FINAL - UNA SOLA APERTURA DE TEST

 Evaluando modelo en TEST (UNA SOLA VEZ):

  MAE en test: 14.93 días
  RMSE en test: 28.08 días

 Comparación con baseline operativo:
  MAE Ingeniero: 29.87 días (basado en 462 muestras válidas)
  MAE Modelo: 14.93 días
  Mejora: 50.0%

 Verificaciones de FUGA en evaluación:
  Test se abrió UNA SOLA VEZ (al final)
  No se usó test para selección de modelo o hiperparámetros
  Todas las decisiones se tomaron con VALIDACIÓN (no test)


### Guardar dataset

In [18]:
# Eliminar columnas temporales antes de guardar
df_final = df.drop(['week_num', 'max_allowed_days', 'is_at_max'], axis=1)

output_file = 'constructora_horizonte_dias_restantes.csv'
df_final.to_csv(output_file, index=False)
print(f"\n Dataset guardado como: {output_file}")

# Guardar diccionario de variables
dict_file = 'diccionario_variables.md'
with open(dict_file, 'w', encoding='utf-8') as f:
    f.write("# Diccionario de Variables - Constructora Horizonte\n\n")
    f.write("## Dataset: Días restantes para completar un frente de trabajo\n\n")
    f.write("| Variable | Tipo | Descripción | Rango/Valores | Clasificación |\n")
    f.write("|----------|------|-------------|---------------|---------------|\n")

    var_dict = {
        'week_date': ['date', 'Fecha de registro semanal', df_final['week_date'].min().strftime('%Y-%m-%d') + ' a ' + df_final['week_date'].max().strftime('%Y-%m-%d'), 'Identificador'],
        'project_id': ['string', 'Identificador único del proyecto', f'{df_final["project_id"].nunique()} proyectos únicos', 'Identificador'],
        'workfront_id': ['string', 'Identificador único del frente', f'{df_final["workfront_id"].nunique()} frentes únicos', 'Identificador'],
        'work_type': ['string', 'Tipo de frente de trabajo', ', '.join(df_final['work_type'].unique().tolist()), 'Predictora'],
        'province': ['string', 'Provincia del proyecto', ', '.join(df_final['province'].unique().tolist()), 'Predictora'],
        'planned_progress_pct': ['float', 'Avance planificado acumulado (%)', f'{df_final["planned_progress_pct"].min():.1f} - {df_final["planned_progress_pct"].max():.1f}', 'Predictora'],
        'actual_progress_pct': ['float', 'Avance real acumulado (%)', f'{df_final["actual_progress_pct"].min():.1f} - {df_final["actual_progress_pct"].max():.1f}', 'Predictora'],
        'workers_assigned': ['int', 'Trabajadores asignados al frente', f'{df_final["workers_assigned"].min()} - {df_final["workers_assigned"].max()}', 'Predictora'],
        'equipment_hours': ['float', 'Horas de equipo utilizadas', f'{df_final["equipment_hours"].min():.1f} - {df_final["equipment_hours"].max():.1f}', 'Predictora'],
        'material_availability_pct': ['float', 'Disponibilidad de materiales (%)', f'{df_final["material_availability_pct"].min():.1f} - {df_final["material_availability_pct"].max():.1f}', 'Predictora'],
        'rain_days_last_week': ['int', 'Días de lluvia semana anterior', f'{df_final["rain_days_last_week"].min()} - {df_final["rain_days_last_week"].max()}', 'Predictora'],
        'change_orders_to_date': ['int', 'Órdenes de cambio acumuladas', f'{df_final["change_orders_to_date"].min()} - {df_final["change_orders_to_date"].max()}', 'Predictora'],
        'contractor_id': ['string', 'Identificador del contratista', f'{df_final["contractor_id"].nunique()} contratistas únicos', 'Predictora'],
        'days_elapsed': ['int', 'Días transcurridos desde inicio', f'{df_final["days_elapsed"].min()} - {df_final["days_elapsed"].max()}', 'Predictora'],
        'engineer_estimate_days': ['int', 'Estimación del ingeniero (baseline operativo)', f'{df_final["engineer_estimate_days"].min()} - {df_final["engineer_estimate_days"].max()}', 'Predictora (Baseline)'],
        'days_remaining': ['int', 'Días reales restantes (TARGET)', f'{df_final["days_remaining"].min()} - {df_final["days_remaining"].max()}', 'Target'],
        'final_completion_date': ['date', '**FUGA**: Fecha real de finalización', df_final['final_completion_date'].notnull().sum(), 'Posterior (Excluir)'],
        'total_final_cost': ['float', '**FUGA**: Costo final del proyecto', f'${df_final["total_final_cost"].min():,.0f} - ${df_final["total_final_cost"].max():,.0f}', 'Posterior (Excluir)'],
        'final_delay_reason': ['string', '**FUGA**: Razón principal del retraso', ', '.join(df_final['final_delay_reason'].dropna().unique().tolist()[:5]), 'Posterior (Excluir)'],
        'final_progress_report': ['string', '**FUGA**: Informe de progreso final', f'{df_final["final_progress_report"].value_counts().to_dict()}', 'Posterior (Excluir)']
    }

    for var, info in var_dict.items():
        f.write(f"| {var} | {info[0]} | {info[1]} | {info[2]} | {info[3]} |\n")

print(f" Diccionario de variables guardado como: {dict_file}")


 Dataset guardado como: constructora_horizonte_dias_restantes.csv
 Diccionario de variables guardado como: diccionario_variables.md


### Resumen final con VERIFICACIONES DE FUGA

In [19]:
print("\n" + "="*70)
print("RESUMEN FINAL - CON VERIFICACIONES DE FUGA")
print("="*70)

print(f"""
ESTADÍSTICAS GENERALES:
  - Total observaciones: {len(df_final):,}
  - Proyectos: {df_final['project_id'].nunique()}
  - Frentes: {df_final['workfront_id'].nunique()}
  - Período: {df_final['week_date'].min().strftime('%Y-%m-%d')} a {df_final['week_date'].max().strftime('%Y-%m-%d')}

TARGET:
  - Variable: days_remaining
  - Media: {df_final['days_remaining'].mean():.1f} días
  - Mediana: {df_final['days_remaining'].median():.0f} días
  - Rango: {df_final['days_remaining'].min()} - {df_final['days_remaining'].max()} días

VERIFICACIONES DE FUGA:
  1. Tope 2× días transcurridos calculado FILA POR FILA
  2. Imputación, escalado, encoding ajustados SOLO en TRAIN
  3. Hiperparámetros seleccionados con VALIDACIÓN (NO test)
  4. Test abierto UNA SOLA VEZ al final
  5. Partición TEMPORAL (NO aleatoria)

ARCHIVOS GENERADOS:
  - {output_file}
  - {dict_file}

GENERACIÓN COMPLETADA CON ÉXITO
Semilla: 2026 | Código original con corrección de fuga
""")


RESUMEN FINAL - CON VERIFICACIONES DE FUGA

ESTADÍSTICAS GENERALES:
  - Total observaciones: 5,428
  - Proyectos: 30
  - Frentes: 150
  - Período: 2025-01-19 a 2025-12-11

TARGET:
  - Variable: days_remaining
  - Media: 207.9 días
  - Mediana: 182 días
  - Rango: 0 - 546 días

VERIFICACIONES DE FUGA:
  1. Tope 2× días transcurridos calculado FILA POR FILA
  2. Imputación, escalado, encoding ajustados SOLO en TRAIN
  3. Hiperparámetros seleccionados con VALIDACIÓN (NO test)
  4. Test abierto UNA SOLA VEZ al final
  5. Partición TEMPORAL (NO aleatoria)

ARCHIVOS GENERADOS:
  - constructora_horizonte_dias_restantes.csv
  - diccionario_variables.md

GENERACIÓN COMPLETADA CON ÉXITO
Semilla: 2026 | Código original con corrección de fuga




#	Verificación	Dónde se aplica	Cómo se implementa
1	Tope 2× días transcurridos	Generación de datos (fila por fila)	max_remaining_days = max(30, days_elapsed * 2) usando SOLO información disponible al momento
2	Imputación ajustada en TRAIN	Preprocesamiento	SimpleImputer(strategy='median') con .fit() en TRAIN, .transform() en TEST
3	Escalado ajustado en TRAIN	Preprocesamiento	StandardScaler con .fit() en TRAIN, .transform() en TEST
4	Encoding ajustado en TRAIN	Preprocesamiento	OneHotEncoder con .fit() en TRAIN, .transform() en TEST
5	Selección de hiperparámetros	Modelado	GridSearchCV con validación cruzada (NO test)
6	Selección de modelo	Modelado	Evaluación con CV (NO test)
7	Evaluación final	Evaluación	Una sola apertura de test al final